
# Just uninstall bson and pymongo and only install pymongo ---then restart the kernel/restart the python---dont install bson ever 

In [0]:
#pip uninstall bson


%pip uninstall -y bson

In [0]:
%pip uninstall -y pymongo

In [0]:
pip install pymongo

In [0]:
dbutils.library.restartPython()

In [0]:
from pymongo import MongoClient
import bson
from bson import json_util

print("Successfully imported!")

In [0]:
# this code cells works perfectly

# I will start from here ----- s3 ingestion ----- homework from uday bhaiya

# here basically we fetch the data from sql server cloud and dump it in raw data folder of databricks

# we used modular architecture in a sense that we created a common functions such as read n write - which will be called for each file

# also we hide our access key using configuration - we did not hardcode it - we used databricks secret scope - to store access keys

# we also created the widgets to pass path of files

# steps are just previous steps , but this time with the use of common utils functions and confugurations - okay


# =======1.IMPORT LIBRARIES & FUNCTIONS=================

import json
from datetime import date
from common_utils.logging import get_logger
import importlib, common_utils.ingestor
importlib.reload(common_utils.ingestor)
from common_utils.ingestor import read_cosmosdb_json, write_raw

logger = get_logger('cosmosdb-ingestion')

# =======2.CREATE WIDGETS FOR PATH=====================
dbutils.widgets.text('path',"")
config_path = dbutils.widgets.get('path')

# =======3.CONFIG VARIABLES FOR ACCESSING KEYS USING CONFIG JSON FILE ===========

with open(config_path,'r') as f:
    config = json.load(f)

source_config = config['source']
target_config = config['target']
write_options_target = config['write_options']

print(source_config)


# =======3.CONFIG VARIABLES FOR ACCESSING KEYS USING CONFIG JSON FILE ===========

with open(config_path,'r') as f:
    config = json.load(f)

source_config = config['source']
target_config = config['target']
write_options_target = config['write_options']

print(source_config)

# =======3.1 RETRIEVE COSMOSDB CONNECTION STRING FROM SECRETS =======

connection_string = dbutils.secrets.get(scope='retail-platform-dev', key='cosmosdb_connection_string')

# ========4.FETCH DATA FROM COSMOSDB ==================

df = read_cosmosdb_json(spark, connection_string, source_config['database_name'], source_config['connection_name'])
logger.info('read %s rows', df.count())
logger.info('sample data looks like.....')
df.show()

# ========5.WRITING THE DATA INTO RAW DATA FOLDER WITH CURRENT DATE FOLDER ====================

from datetime import date
run_date = date.today().isoformat()
logger.info('load date is %s', run_date)

target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"
logger.info("writiing data to %s", target_path)

target_path = write_raw(df, target_path,target_config["file_format"], target_config["mode"], None)
logger.info("data landed at %s", target_path)

In [0]:



# ========5.WRITING THE DATA INTO RAW DATA FOLDER WITH CURRENT DATE FOLDER ====================

from datetime import date
run_date = date.today().isoformat()
logger.info('load date is %s', run_date)

target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"
logger.info("writiing data to %s", target_path)

target_path = write_raw(df, target_path,target_config["file_format"], target_config["mode"], None)
logger.info("data landed at %s", target_path)



# code works from here - newly added

In [0]:
from datetime import date 

connection_string = dbutils.secrets.get(scope='retail-platform-dev', key='cosmosdb_connection_string') 
database_name = "retail" 
collection_name = "sales_orders" 

from pymongo import MongoClient
from bson import json_util


client = MongoClient(connection_string) 
db = client[database_name] 

collection = db[collection_name] 
documents = list(collection.find({})) 

print(len(documents)) 
json_rows = [] 

for document in documents: 
    json_string = json_util.dumps(document) 
    json_rows.append((json_string,)) 
    

df = spark.createDataFrame(json_rows, ["json_data"]) 


In [0]:

run_date = date.today().isoformat() 
target_path = f"/Volumes/retaildataplatform/bronze/raw_data/cosmosdb_sales_orders/test/load_date={run_date}" 

df.write.format("json").mode("overwrite").save(target_path)



In [0]:
# we will do same for cosmosdb

# 1.read the data from raw path n display the data
# 2. add audit columns and show the data with added columns
# 3. define target path - bronze.orders
# 4. write the data to target path


In [0]:
raw_path = "/Volumes/retaildataplatform/bronze/raw_data/cosmosdb_sales_orders/test/load_date=2026-09-11/"

df = spark.read.json(raw_path)

df.display()

In [0]:
import pyspark.sql.functions as F

from datetime import date

print('add audit columns:')
df = df.withColumn('timestamp_df_column', F.current_timestamp())
#df = df.withColumn('file_path_new_column', F.col('_metadata.file_path'))

# to show newly added columns
df.display()

In [0]:
df = df.withColumn('file_path_new_column', F.col('_metadata.file_path'))

# to show newly added columns
df.display()

In [0]:
target_path = "retaildataplatform.bronze.cosmos_sales_orders"
df.write.mode('overwrite').format('delta').saveAsTable(target_path)